## Visualization/Case Study Primer

The intention of this notebook is to provide general guidelines for using the Python visualization package matplotlib. You will get some exposure to the types of plots you can create, the coding structure to produce plots and some tools/tips to make your life a little easier. If you can't find what you're looking for in here, using an LLM to vibe code a solution is also completely fine!!! Lastly, feel free to reach out to me (jonchann@my.yorku.ca) if you have questions, comments or concerns. 

Without further ado, let's get started by importing some packages and a dataset to work with.

In [ ]:
import matplotlib.pyplot as plt # visualization package
import plotly.express as px # visualization package for geogaphical plotting
import numpy as np # numerical package for generating data
import pandas as pd # data manipulation package
from shapely.geometry import Point
import geopandas as gpd
import json
import warnings
warnings.filterwarnings('ignore')
# You'll need to have these packages installed in your environment before you run this notebook
# pip install {package_name}, in your command line

In [ ]:
pip install shapely 

In [ ]:
pip install geopandas 

In [ ]:
# read our data set in, we'll use the dinesafe data from Open Data Toronto

df = pd.read_csv("Dinesafe.csv")

df

In [ ]:
print(df["Establishment Type"].unique()) # lets get just restaurants
print(df["Establishment Status"].value_counts()) # Not many conditional passes handed out, 
restaurants = df[df["Establishment Type"] == "Restaurant"]

# immediately see that there there are multiple entries for each inspection if there were multiple infractions
# will inflate raw inspection counts

In [ ]:
restaurants.dtypes

In [ ]:
# checking NaN values, can mess with plotting if not handled correctly
# Infraction details, severity, action, outcome and amount fined are likely NaNs because the inspector had nothing to comment on
# Inspection ID will not be plotted
# Inspection date might be used, but it has ~.4210% missing data so we will not do anything yet
restaurants.isna().sum()/len(restaurants)

In [ ]:
for col in restaurants.columns:
    missing_ratio = restaurants[col].isna().sum() / len(restaurants)

    if missing_ratio > 0.25:
        if restaurants[col].dtype == "object":
            restaurants[col] = restaurants[col].fillna("None")
        elif restaurants[col].dtype == "float64":
            restaurants[col] = restaurants[col].fillna(0)

restaurants.isna().sum()/len(restaurants)
# Much better. Let's do some quick visualizations

## Single Variable Plots

These are the simplest plots to construct. They require only one variable and they plot things like frequencies, counts and trends over time. I will first show you the general structure of a matplotlib visualization that you can follow reliably to generate a variety of plots. The two plots featured here are the NaN percentages per column for the original dataframe and multi-line plot showing the overall amount of inspections per year binned by month.

In [ ]:
# plotting the NaN percentages of the original dataframe

primer = (df.isna().sum()/len(df)).sort_values() # The data we're plotting

fig, ax = plt.subplots() # you can specify multiple plots and figure size here too. You'll see it used later on.

ax.barh(primer.index, primer.values) # creates a horizontal bar plot, can be other types of plots like a line plot (plt.plot)
ax.set_title("Percentage of NaNs") # Title of plot
ax.set_xlabel("NaN Percentages per Column") # X-axis title
ax.set_ylabel("Column Name") # Y-axis title
plt.show() # Always use this

# There are several more options that can be specified, but this is the general basic structure you will want to follow when making plots

In [ ]:
restaurants["Inspection Date"] = pd.to_datetime(
    restaurants["Inspection Date"], format="%Y-%m-%d", errors="coerce"
) # change dates to datetime objects, easier to work with

dedup = restaurants.drop_duplicates(
    subset=["Establishment Name", "Inspection Date"]
).copy() # remove duplicates to get one entry per inspection, otherwise we are countting innacurately

dedup["Year"] = dedup["Inspection Date"].dt.year
dedup["Month"] = dedup["Inspection Date"].dt.month

monthly_by_year = dedup.groupby(["Year", "Month"]).size().unstack(fill_value=0)

In [ ]:
fig, ax = plt.subplots(figsize = (12,8))

for year in monthly_by_year.index:
    ax.plot(monthly_by_year.columns, monthly_by_year.loc[year], label=str(year))

ax.set_title("Monthly Inspections per Year")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Inspections")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])

ax.legend(title="Year", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

# Inspection counts per year, binned by month

## Geographical Plotting

Here you'll see a quick example of the kind of plots you can make using Python packages to inspire your collages. This will require to install packages such as geopandas or plotly. I will be using both of the mentioned, but feel free to use any package that you like.

First we will start with plotting all restaurants and their Dinesafe severity scores using plotly. All the plots frrom now arer going to be interactive. 

In [ ]:
fig = px.scatter_mapbox( #scatter plot on map
    restaurants, # dataframe
    lat = "Latitude", 
    lon = "Longitude",
    hover_name = "Establishment Name", # Title when you hover over a point
    hover_data = "Severity", # info you want included in the hover info
    color = "Severity", # color coding for points, also included in the hover info
    zoom = 11, # starting zoom
    height = 600 # size
)

fig.update_layout(
    mapbox_style = "open-street-map", # map type
    mapbox_center = {"lat": 43.65107, "lon": -79.347015} # centering
)

fig.show()
# Another reason why visuals are important is that we found an extra category in severity. Let's deal with that.

What the above plot showed us is that we needed more cleaning! Below are those steps and a follow up plot where all the categories are plotted and we added an animation slider for yyear to see the year to year changes in severity.

In [ ]:
restaurants["Severity"] = restaurants["Severity"].str.strip() # more cleaning, adding consistency to strings for manipulation
restaurants["Severity"] = restaurants["Severity"].fillna("None")

restaurants.loc[
    restaurants["Severity"] == "NA - Not Applicable", 
    "Severity"
] = "None" # mapping the unexpected category to None

In [ ]:
severity_order = ["None", "M - Minor", "S - Significant", "C - Crucial"]
severity_to_score = {
    "None": 0,
    "M - Minor": 1,
    "S - Significant": 2,
    "C - Crucial": 3
} # score mapping to integers for easier caetgorical analysis

score_to_severity = {v: k for k, v in severity_to_score.items()}

# ordered categorical
restaurants["Severity"] = pd.Categorical(
    restaurants["Severity"],
    categories=severity_order,
    ordered=True
)

# numeric severity for aggregation
restaurants["SeverityScore"] = restaurants["Severity"].map(severity_to_score).astype(float)

# year column
restaurants["Year"] = restaurants["Inspection Date"].dt.year

# worst severity per inspection
inspection_point_level = (
    restaurants.groupby(
        ["Inspection ID", "Establishment Name", "Latitude", "Longitude", "Year"],
        as_index=False
    )
    .agg(
        WorstSeverityScore=("SeverityScore", "max")
    )
)

# average worst inspection severity per restaurant-year
restaurant_year = (
    inspection_point_level.groupby(
        ["Establishment Name", "Latitude", "Longitude", "Year"],
        as_index=False
    )
    .agg(
        AvgWorstSeverityScore=("WorstSeverityScore", "mean"),
        InspectionCount=("WorstSeverityScore", "size"),
        MaxWorstSeverityScore=("WorstSeverityScore", "max")
    )
)

# rounded category for discrete coloring
restaurant_year["SeverityLabel"] = pd.Categorical(
    restaurant_year["AvgWorstSeverityScore"].round().clip(0, 3).astype(int).map(score_to_severity),
    categories=severity_order,
    ordered=True
)

In [ ]:
fig = px.scatter_mapbox(
    restaurant_year,
    lat="Latitude",
    lon="Longitude",
    hover_name="Establishment Name",
    hover_data={
        "SeverityLabel": True,
        "AvgWorstSeverityScore": ':.2f',
        "InspectionCount": True,
        "MaxWorstSeverityScore": True,
        "Latitude": False,
        "Longitude": False
    },
    color="SeverityLabel",
    category_orders={"SeverityLabel": severity_order},
    animation_frame="Year",
    zoom=11,
    height=600
)

fig.update_layout(
    mapbox_style="open-street-map",
    mapbox_center={"lat": 43.65107, "lon": -79.347015}
)

fig.show()

Looking much better now. Everything appearrs to be in the right place and our data is cleaned enough to generate some more meaningful plots. From here on out, you will see some plots and ideas that you may be interrested in doing yourself. I will detail all the steps I take and the type of plots I generate.

In [ ]:
severity_heatmap = inspection_point_level.dropna(
    subset=["Latitude", "Longitude", "WorstSeverityScore"]
).copy()

severity_heatmap["HeatSeverityScore"] = (
    (severity_heatmap["WorstSeverityScore"] - 1).clip(lower=0) / 2
)

fig = px.density_mapbox(
    severity_heatmap,
    lat="Latitude",
    lon="Longitude",
    z="HeatSeverityScore",
    hover_data={
        "Latitude": False,
        "Longitude": False,
        "Establishment Name": True,
        "Inspection ID": True,
        "WorstSeverityScore": True,
        "HeatSeverityScore": False
    },
    animation_frame="Year",
    radius=10,
    center={"lat": 43.65107, "lon": -79.347015},
    zoom=10,
    height=650,
    color_continuous_scale="YlOrRd"
)

fig.update_layout(
    mapbox_style="open-street-map",
    annotations=[
        dict(
            text="Metric: point-level severity density for scores above 1, with reduced smoothing",
            x=.96,
            y=0.02,
            xref="paper",
            yref="paper",
            xanchor="right",
            yanchor="bottom",
            showarrow=False,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(0,0,0,0.25)",
            borderwidth=1,
            font=dict(size=11)
        )
    ]
)

fig.show()

## Grid Map

Next we will do a grid map that measures both the amount of inspections in an area of the city, and it will be colour coded based on the average severity of the infractions located in that part of the city. To create this plot, you'll need to specify the uniform grid sectors yourself like I did below. These kinds of plots add a lot more spacial structure than a heatmap, but you have to be careful with how you bin your data. Too small and your plot will be too difficult to read alonside not having enough information for accurate measurements in low density areas. Too large and you lose the spatial coherence since items that shouldn't be lumped together are.

In [ ]:
grid_size = 0.005

restaurant_year["lat_bin"] = (restaurant_year["Latitude"] / grid_size).round() * grid_size
restaurant_year["lon_bin"] = (restaurant_year["Longitude"] / grid_size).round() * grid_size

grid_summary = (
    restaurant_year.groupby(["Year", "lat_bin", "lon_bin"], as_index=False)
    .agg(
        RestaurantCount=("Establishment Name", "nunique"),
        InspectionCount=("InspectionCount", "sum"),
        AvgWorstSeverityScore=("AvgWorstSeverityScore", "mean"),
        MaxWorstSeverityScore=("MaxWorstSeverityScore", "max")
    )
)

grid_summary

In [ ]:
fig = px.scatter_mapbox(
    grid_summary,
    lat="lat_bin",
    lon="lon_bin",
    color="AvgWorstSeverityScore",
    size="RestaurantCount",
    hover_data={
        "lat_bin": False,
        "lon_bin": False,
        "RestaurantCount": True,
        "InspectionCount": True,
        "AvgWorstSeverityScore": ":.2f",
        "MaxWorstSeverityScore": True
    },
    animation_frame="Year",
    color_continuous_scale="Viridis",
    zoom=10,
    height=700
)

fig.update_layout(mapbox_style="open-street-map")

fig.show()

## Choropleth Map

This final visualization is intended for you to see what yor "final" visualization could look like before creating your collages. For this example, I needed to import a neighbourhood .geojson file to map inspections to their respective neighbourhoods using their latitude and longitude ([link](https://github.com/jasonicarter/toronto-geojson)). Before we visualize, we need to aggregate our data based on neighbourhood and incluide our severity scores to generate meaningful visuals. Choropleth maps are powerful visualization tools for geographical data. In this case, you can easily find your neighborhood simply by hovering over it to see the average infraction score and how many inspections occur on a year to year basis.

In [ ]:
restaurants_gdf = gpd.GeoDataFrame(
    restaurants.copy(),
    geometry=gpd.points_from_xy(restaurants["Longitude"], restaurants["Latitude"]),
    crs="EPSG:4326"
)
# creating a geopandas dataframe. maps long/lat to a map

neighbourhoods = gpd.read_file("toronto_crs84.geojson")
# neighbourhood file using the .geojson file

neighbourhoods = neighbourhoods.to_crs(restaurants_gdf.crs)
# maps coordinate system from our geopandas dataframe into the neighbourhood coordinates system

joined = gpd.sjoin(
    restaurants_gdf,
    neighbourhoods,
    how="left",
    predicate="within"
)
# join the two dataframes

inspection_level = (
    joined.groupby(["Inspection ID", "AREA_NAME", "Year"], as_index=False)
    .agg(
        MaxSeverity=("SeverityScore", "max")
    )
)
# get max severity per inspection

neigh_summary = (
    inspection_level.groupby(["AREA_NAME", "Year"], as_index=False)
    .agg(
        Count=("MaxSeverity", "size"),
        AvgSeverity=("MaxSeverity", "mean"),
        MaxSeverity=("MaxSeverity", "max")
    )
)
# summary table that groups by the neighbourhood and year, and aggregates the data into counts, average severity score and max severity score

map_df = neighbourhoods.merge(neigh_summary, on="AREA_NAME", how="left")

map_df
# final merge

In [ ]:
map_json = json.loads(map_df.to_json())

fig = px.choropleth_mapbox(
    map_df,
    geojson = map_json,
    locations = map_df.index,
    color = "AvgSeverity",
    range_color = (map_df["AvgSeverity"].min(), map_df["AvgSeverity"].max()),
    hover_name = "AREA_NAME",
    hover_data = {
        "Count": True,
        "AvgSeverity": ':.2f',
        "MaxSeverity": True
    },
    animation_frame = "Year",
    center = {"lat": 43.6532, "lon": -79.3832},
    zoom = 9,
    mapbox_style = "open-street-map",
    opacity = 0.6,
    height = 700
)

fig.update_layout(margin={"r": 0, "t": 30, "l": 0, "b": 0})
fig.show()
# update per year slider, start at aggregation